# 连接配置

In [ ]:
import json
import os
from pathlib import Path

import tushare as ts

# 本地配置不提交 Git；环境变量的优先级更高。
local_config_path = Path.cwd() / ".local" / "data_source.json"
local_config = json.loads(local_config_path.read_text(encoding="utf-8-sig")) if local_config_path.is_file() else {}
TOKEN = os.environ.get("TUSHARE_TOKEN") or local_config.get("tushare_token")
if not TOKEN:
    raise RuntimeError("请设置 TUSHARE_TOKEN 环境变量，或在本地 .local/data_source.json 中配置 tushare_token。")
pro = ts.pro_api(TOKEN)
api_url = os.environ.get("TUSHARE_API_URL") or local_config.get("tushare_api_url")
if api_url:
    pro._DataApi__http_url = api_url


# 安装依赖

In [ ]:
%pip install pandas tqdm

# 创建data

In [ ]:
from pathlib import Path
import time

import pandas as pd
from tqdm.auto import tqdm

# 沿用你之前测试成功的 pro 对象
if "pro" not in globals():
    raise RuntimeError("请先运行创建 pro 的连接配置单元格")

PROJECT_DIR = Path.cwd()

START_DATE = "20220101"
END_DATE = "20251231"

# 请求之间的间隔；如果服务限频，可调大
REQUEST_INTERVAL = 1.0

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

DIRS = {
    "basic": RAW_DIR / "basic",
    "daily": RAW_DIR / "daily",
    "adj_factor": RAW_DIR / "adj_factor",
    "daily_basic": RAW_DIR / "daily_basic",
    "benchmark": RAW_DIR / "benchmark",
    "checks": DATA_DIR / "checks",
    "processed": DATA_DIR / "processed",
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("项目目录：", PROJECT_DIR)
print("数据目录：", DATA_DIR)
print("下载区间：", START_DATE, "至", END_DATE)

通用下载与保存函数
这段代码负责：
- 失败后重试。
- 检查返回字段。
- 先写临时文件，成功后再替换。
- 检查已有 CSV，支持续传。
- 遇到疑似接口截断时停止，而不是保存不完整数据。

In [ ]:
def request_data(api_name, **kwargs):
    """调用接口，失败最多重试3次。"""
    for attempt in range(3):
        try:
            time.sleep(REQUEST_INTERVAL)

            df = getattr(pro, api_name)(**kwargs)

            if not isinstance(df, pd.DataFrame):
                raise TypeError(f"{api_name} 返回的不是 DataFrame")

            return df

        except Exception as exc:
            if attempt == 2:
                raise

            print(
                f"{api_name} 请求失败，第 {attempt + 1} 次重试："
                f"{type(exc).__name__}"
            )
            time.sleep(3 * (attempt + 1))


def save_csv(df, path):
    """完整写入后替换正式文件，避免中断留下半个CSV。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_suffix(".tmp")
    df.to_csv(temp_path, index=False, encoding="utf-8-sig")
    temp_path.replace(path)


def validate_table(df, required_columns, keys=None):
    if df.empty:
        raise ValueError("返回空数据，不能视为下载成功")

    missing = set(required_columns) - set(df.columns)
    if missing:
        raise ValueError(f"缺少字段：{sorted(missing)}")

    if keys:
        if df[keys].isna().any().any():
            raise ValueError(f"主键存在空值：{keys}")

        if df.duplicated(keys).any():
            raise ValueError(f"主键重复：{keys}")


def cached_daily_ok(path, required_columns, trade_date):
    """检查已有文件是否基本有效，而不只是判断是否存在。"""
    if not path.exists():
        return False

    try:
        df = pd.read_csv(path, dtype=str)

        validate_table(
            df,
            required_columns,
            keys=["ts_code", "trade_date"]
        )

        return df["trade_date"].eq(trade_date).all()

    except Exception:
        return False

下载股票基础信息和交易日历
股票基础信息按不同状态分别请求，避免只保留现在仍上市的股票。

# ==================================================
# 下载股票基础信息与交易日历
# ==================================================


In [ ]:

import pandas as pd
from IPython.display import display

# 检查前置配置
required_variables = [
    "pro", "request_data", "save_csv",
    "DIRS", "START_DATE", "END_DATE"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        f"请先运行配置与通用函数单元格，缺少：{missing_variables}"
    )




# --------------------------------------------------
# 1. 分状态下载股票基础信息，保存原始返回
# --------------------------------------------------

In [ ]:


status_tables = {}
parts = []

for status, label in [
    ("L", "上市"),
    ("D", "退市"),
    ("P", "暂停上市")
]:
    part = request_data(
        "stock_basic",
        exchange="",
        list_status=status,
        fields=(
            "ts_code,symbol,name,area,industry,market,"
            "exchange,list_status,list_date,delist_date"
        )
    )

    print(f"{label}：{len(part)} 行")

    if part.empty:
        continue

    if "ts_code" not in part.columns:
        raise ValueError(f"{label}数据缺少 ts_code")

    # 原始返回单独保存
    save_csv(
        part,
        DIRS["basic"] / f"stock_basic_raw_{status}.csv"
    )

    part = part.copy()

    if part["ts_code"].isna().any():
        raise ValueError(f"{label}数据存在空股票代码")

    part["ts_code"] = (
        part["ts_code"].astype("string").str.strip()
    )

    if part["ts_code"].eq("").any():
        raise ValueError(f"{label}数据存在空字符串股票代码")

    # 这是请求条件，不代表已经核实的真实状态
    part["requested_list_status"] = status

    status_tables[status] = part.copy()
    parts.append(part)

if not parts:
    raise RuntimeError("股票基础信息全部为空，请检查接口")

raw_basic = pd.concat(parts, ignore_index=True)




# --------------------------------------------------
# 2. 保存重复记录，保留冲突信息
# --------------------------------------------------


In [ ]:

duplicate_records = raw_basic.loc[
    raw_basic.duplicated("ts_code", keep=False)
].sort_values(["ts_code", "requested_list_status"])

save_csv(
    duplicate_records,
    DIRS["checks"] / "stock_basic_duplicate_records.csv"
)

print(
    "\n重复代码数量：",
    duplicate_records["ts_code"].nunique()
)

if not duplicate_records.empty:
    display(duplicate_records)




# --------------------------------------------------
# 3. 筛选沪深A股代码
# 暂不纳入北交所、B股及其他证券
# --------------------------------------------------


In [ ]:

codes = raw_basic["ts_code"]

is_sh_a = codes.str.fullmatch(
    r"(600|601|603|605|688)\d{3}\.SH"
)

is_sz_a = codes.str.fullmatch(
    r"(000|001|002|003|300|301)\d{3}\.SZ"
)

a_share_records = raw_basic.loc[
    (is_sh_a | is_sz_a).fillna(False)
].copy()

if a_share_records.empty:
    raise RuntimeError("筛选后没有沪深A股，请检查股票代码格式")

# 保留所有返回记录，不擅自选择冲突记录
save_csv(
    a_share_records,
    DIRS["basic"] / "stock_basic_a_share_records.csv"
)

# 每只股票一行，汇总它出现在哪些请求状态中
stock_codes = (
    a_share_records
    .groupby("ts_code", as_index=False)
    .agg(
        returned_in_statuses=(
            "requested_list_status",
            lambda values: "|".join(
                sorted(set(values.dropna()))
            )
        )
    )
)

stock_codes["status_conflict"] = (
    stock_codes["returned_in_statuses"]
    .str.contains("|", regex=False)
)

stock_codes = (
    stock_codes
    .sort_values("ts_code")
    .reset_index(drop=True)
)

save_csv(
    stock_codes,
    DIRS["basic"] / "stock_codes_a_share.csv"
)

print("\n沪深A股代码数量：", len(stock_codes))
print("跨状态冲突数量：", stock_codes["status_conflict"].sum())

display(stock_codes.head())




# --------------------------------------------------
# 4. 下载交易日历
# --------------------------------------------------

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

required = [
    "pro", "request_data", "save_csv",
    "DIRS", "START_DATE", "END_DATE"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"请先运行前面的配置和函数，缺少：{missing}")

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


def normalize_date_column(df, column):
    """将日期统一为YYYYMMDD字符串。"""
    df = df.copy()

    df[column] = pd.to_datetime(
        df[column].astype("string"),
        format="%Y%m%d",
        errors="raise"
    ).dt.strftime("%Y%m%d")

    if df[column].isna().any():
        raise ValueError(f"{column} 存在空日期")

    return df


calendar_parts = []
benchmark_parts = []

years = range(int(START_DATE[:4]), int(END_DATE[:4]) + 1)

for year in tqdm(years, desc="日历与基准"):
    begin = max(START_DATE, f"{year}0101")
    finish = min(END_DATE, f"{year}1231")

    # 1. 请求交易日历
    cal = request_data(
        "trade_cal",
        exchange="SSE",
        start_date=begin,
        end_date=finish,
        fields="exchange,cal_date,is_open,pretrade_date"
    )

    if cal.empty:
        raise ValueError(f"{year} 年交易日历为空")

    if not {"cal_date", "is_open"}.issubset(cal.columns):
        raise ValueError(
            f"交易日历缺少必要字段，实际字段：{cal.columns.tolist()}"
        )

    # 先保留原始返回
    save_csv(
        cal,
        DIRS["basic"] / f"trade_calendar_raw_{begin}_{finish}.csv"
    )

    cal = normalize_date_column(cal, "cal_date")

    cal["is_open"] = pd.to_numeric(cal["is_open"], errors="raise")

    if not cal["is_open"].isin([0, 1]).all():
        raise ValueError("is_open 存在非0、1的值")

    if cal["cal_date"].duplicated().any():
        raise ValueError(f"{year} 年交易日历有重复日期")

    if not cal["cal_date"].between(begin, finish).all():
        raise ValueError(f"{year} 年交易日历返回区间外日期")

    calendar_parts.append(cal)

    # 2. 请求沪深300价格指数行情
    benchmark = request_data(
        "index_daily",
        ts_code="000300.SH",
        start_date=begin,
        end_date=finish,
        fields=(
            "ts_code,trade_date,open,high,low,close,"
            "pre_close,change,pct_chg,vol,amount"
        )
    )

    if benchmark.empty:
        raise ValueError(f"{year} 年沪深300行情为空")

    needed = {"ts_code", "trade_date", "close"}

    if not needed.issubset(benchmark.columns):
        raise ValueError(
            f"指数行情缺少必要字段：{needed - set(benchmark.columns)}"
        )

    benchmark = normalize_date_column(benchmark, "trade_date")

    if benchmark.duplicated(["ts_code", "trade_date"]).any():
        raise ValueError(f"{year} 年指数行情有重复记录")

    if not benchmark["ts_code"].eq("000300.SH").all():
        raise ValueError("指数接口返回了其他证券代码")

    if not benchmark["trade_date"].between(begin, finish).all():
        raise ValueError("指数接口返回区间外日期")

    benchmark_close = pd.to_numeric(
        benchmark["close"], errors="raise"
    )

    if not (benchmark_close.notna() & benchmark_close.gt(0)).all():
        raise ValueError("指数收盘价存在缺失或非正值")

    save_csv(
        benchmark.sort_values("trade_date"),
        DIRS["benchmark"] / f"csi300_{begin}_{finish}.csv"
    )

    benchmark_parts.append(benchmark)


# 3. 合并并交叉核对交易日期
calendar = pd.concat(calendar_parts, ignore_index=True)
benchmark_all = pd.concat(benchmark_parts, ignore_index=True)

if calendar["cal_date"].duplicated().any():
    raise ValueError("合并后的日历日期重复")

if benchmark_all["trade_date"].duplicated().any():
    raise ValueError("合并后的基准日期重复")

calendar_open_dates = set(
    calendar.loc[calendar["is_open"].eq(1), "cal_date"]
)

benchmark_dates = set(benchmark_all["trade_date"])

if calendar_open_dates != benchmark_dates:
    only_calendar = sorted(calendar_open_dates - benchmark_dates)
    only_benchmark = sorted(benchmark_dates - calendar_open_dates)

    raise ValueError(
        "日历与基准交易日期不一致，先不要批量下载。\n"
        f"仅日历有：{only_calendar[:10]}\n"
        f"仅基准有：{only_benchmark[:10]}"
    )

trade_dates = sorted(calendar_open_dates)

if not trade_dates:
    raise ValueError("交易日列表为空")

# 不擅自把接口没返回的日期填成休市日
calendar = calendar.sort_values("cal_date").reset_index(drop=True)

save_csv(
    calendar,
    DIRS["basic"] / f"trade_calendar_{START_DATE}_{END_DATE}.csv"
)

save_csv(
    pd.DataFrame({"trade_date": trade_dates}),
    DIRS["basic"] / f"trade_dates_{START_DATE}_{END_DATE}.csv"
)

save_csv(
    benchmark_all.sort_values("trade_date"),
    DIRS["benchmark"] / f"csi300_{START_DATE}_{END_DATE}.csv"
)

print("日历与指数行情的交易日期一致")
print("日历返回行数：", len(calendar))
print("确认的交易日数量：", len(trade_dates))
print("首个交易日：", trade_dates[0])
print("最后交易日：", trade_dates[-1])

# 定义每日数据

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# 检查前面的配置是否已运行
required = [
    "trade_dates", "request_data", "save_csv",
    "normalize_date_column", "DIRS"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"请先运行前面的单元格，缺少：{missing}")

if not trade_dates:
    raise RuntimeError("交易日列表为空")

# 需要下载的字段
DAILY_TASKS = {
    "daily": [
        "ts_code", "trade_date",
        "open", "high", "low", "close", "pre_close",
        "pct_chg", "vol", "amount"
    ],

    "adj_factor": [
        "ts_code", "trade_date", "adj_factor"
    ],

    "daily_basic": [
        "ts_code", "trade_date",
        "turnover_rate", "pe_ttm", "pb",
        "total_mv", "circ_mv"
    ]
}

# 疑似返回行数上限检查
# 自定义服务若有更低上限，应按该服务文档调整
ROW_LIMIT_GUARD = 6000


def daily_file_path(api_name, date):
    """生成文件路径，例如 data/raw/daily/2022/20220104.csv。"""
    return DIRS[api_name] / date[:4] / f"{date}.csv"


def check_daily_table(df, api_name, date):
    """检查字段、日期、重复记录和基本数值。"""
    if df.empty:
        raise ValueError(f"{api_name} {date} 返回空数据")

    missing_columns = set(DAILY_TASKS[api_name]) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{api_name} 缺少字段：{sorted(missing_columns)}"
        )

    df = normalize_date_column(df, "trade_date")

    if df["ts_code"].isna().any():
        raise ValueError(f"{api_name} 存在空股票代码")

    df["ts_code"] = df["ts_code"].astype("string").str.strip()

    if df["ts_code"].eq("").any():
        raise ValueError(f"{api_name} 存在空字符串股票代码")

    if df.duplicated(["ts_code", "trade_date"]).any():
        raise ValueError(f"{api_name} 存在重复的股票与日期")

    if not df["trade_date"].eq(date).all():
        raise ValueError(f"{api_name} 返回了其他日期的数据")

    if len(df) >= ROW_LIMIT_GUARD:
        raise ValueError(
            f"{api_name} 返回 {len(df)} 行，可能触及返回上限，"
            "需要检查服务端限制"
        )

    numeric_columns = [
        col for col in DAILY_TASKS[api_name]
        if col not in ["ts_code", "trade_date"]
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="raise")

        if df[col].isna().all():
            raise ValueError(f"{api_name} 的 {col} 整列为空")

    positive_columns = {
        "daily": ["open", "high", "low", "close"],
        "adj_factor": ["adj_factor"],
        "daily_basic": []
    }[api_name]

    for col in positive_columns:
        if not (df[col].notna() & df[col].gt(0)).all():
            raise ValueError(
                f"{api_name} 的 {col} 存在缺失或非正值"
            )

    return df


def read_valid_cache(api_name, date):
    """读取已有文件；文件不存在或检查失败时返回None。"""
    path = daily_file_path(api_name, date)

    if not path.exists():
        return None

    try:
        df = pd.read_csv(
            path,
            dtype={"ts_code": str, "trade_date": str}
        )

        return check_daily_table(df, api_name, date)

    except Exception:
        return None


def download_one_day(api_name, date):
    """下载一天的数据，已有合格文件则跳过请求。"""
    cached = read_valid_cache(api_name, date)

    if cached is not None:
        return cached, "cached"

    df = request_data(
        api_name,
        trade_date=date,
        fields=",".join(DAILY_TASKS[api_name])
    )

    df = check_daily_table(df, api_name, date)

    save_csv(
        df.sort_values("ts_code"),
        daily_file_path(api_name, date)
    )

    return df, "downloaded"


print("每日下载函数准备完成")
print("待下载交易日数量：", len(trade_dates))

In [ ]:
test_date = trade_dates[0]

for api_name in DAILY_TASKS:
    df_test, status = download_one_day(api_name, test_date)

    print(
        f"\n{api_name} | {test_date} | "
        f"{status} | {len(df_test)} 行"
    )

    display(df_test.head(3))

    print("字段缺失比例：")
    display(df_test.isna().mean())

In [ ]:
print("当前请求间隔：", REQUEST_INTERVAL, "秒")

# 只检查超时配置，不输出 Token 或其他连接信息
client_config = vars(pro)
timeout_key = "_DataApi__timeout"

if timeout_key in client_config:
    old_timeout = client_config[timeout_key]
    print("当前客户端超时：", old_timeout, "秒")

    # 仅在已有超时设置小于30秒时调整
    if isinstance(old_timeout, (int, float)) and 0 < old_timeout < 30:
        setattr(pro, timeout_key, 30)
        print("客户端超时已调整为30秒")
    else:
        print("保留当前超时设置")
else:
    print("未识别到标准 Tushare 超时配置，本格未修改客户端")

In [ ]:
for api_name in DAILY_TASKS:
    print(f"\n开始下载：{api_name}")

    for date in tqdm(trade_dates, desc=api_name):
        try:
            download_one_day(api_name, date)

        except Exception:
            save_csv(
                pd.DataFrame([{
                    "api": api_name,
                    "trade_date": date,
                    "status": "failed"
                }]),
                DIRS["checks"] / "last_failed_task.csv"
            )

            print(
                f"\n下载停止：{api_name}，日期：{date}。\n"
                "已保存的数据保留，修复后重新运行本单元格即可。"
            )
            raise

print("\n全部每日下载任务结束，请继续检查文件。")

Check

In [ ]:
file_rows = []
coverage_rows = []

for date in tqdm(trade_dates, desc="检查下载结果"):
    tables = {}

    for api_name in DAILY_TASKS:
        df = read_valid_cache(api_name, date)
        valid = df is not None

        file_rows.append({
            "trade_date": date,
            "api": api_name,
            "valid": valid,
            "rows": len(df) if valid else 0
        })

        if valid:
            tables[api_name] = df

    # 检查行情中的股票能否匹配复权因子和每日指标
    if len(tables) == len(DAILY_TASKS):
        daily_codes = set(tables["daily"]["ts_code"])

        for other in ["adj_factor", "daily_basic"]:
            missing_codes = sorted(
                daily_codes - set(tables[other]["ts_code"])
            )

            coverage_rows.append({
                "trade_date": date,
                "table": other,
                "missing_count": len(missing_codes),
                "examples": "|".join(missing_codes[:10])
            })

file_checks = pd.DataFrame(file_rows)

coverage_checks = pd.DataFrame(
    coverage_rows,
    columns=["trade_date", "table", "missing_count", "examples"]
)

save_csv(
    file_checks,
    DIRS["checks"] / f"file_check_{START_DATE}_{END_DATE}.csv"
)

save_csv(
    coverage_checks,
    DIRS["checks"] / f"coverage_check_{START_DATE}_{END_DATE}.csv"
)

print("文件检查汇总：")
display(
    file_checks.groupby("api").agg(
        expected_days=("trade_date", "count"),
        valid_days=("valid", "sum"),
        min_rows=("rows", "min"),
        max_rows=("rows", "max")
    )
)

invalid_files = file_checks.loc[~file_checks["valid"]]

print("缺失或无效文件数量：", len(invalid_files))
display(invalid_files.head(10))

coverage_problems = coverage_checks.loc[
    coverage_checks["missing_count"].gt(0)
]

print("存在跨表匹配缺口的记录数：", len(coverage_problems))
display(coverage_problems.head(10))

合并一天的数据并保存预览

In [ ]:
date = trade_dates[-1]

tables = {
    api_name: read_valid_cache(api_name, date)
    for api_name in DAILY_TASKS
}

if any(df is None for df in tables.values()):
    raise RuntimeError(f"{date} 的数据文件不完整，请先补下载")

# 合并行情、复权因子、每日指标
merged = (
    tables["daily"]
    .merge(
        tables["adj_factor"],
        on=["ts_code", "trade_date"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        tables["daily_basic"],
        on=["ts_code", "trade_date"],
        how="left",
        validate="one_to_one"
    )
)

# 筛选沪深A股
# 不用当前上市名单过滤历史行情，避免遗漏已退市股票
codes = merged["ts_code"].astype("string")

is_sh_a = codes.str.fullmatch(
    r"(600|601|603|605|688)\d{3}\.SH"
)

is_sz_a = codes.str.fullmatch(
    r"(000|001|002|003|300|301)\d{3}\.SZ"
)

merged = merged.loc[
    (is_sh_a | is_sz_a).fillna(False)
].copy()

preview_path = (
    DIRS["processed"] / f"market_preview_{date}.csv"
)

save_csv(merged, preview_path)

print("预览日期：", date)
print("沪深A股行数：", len(merged))
print("复权因子缺失数：", merged["adj_factor"].isna().sum())
print("总市值缺失数：", merged["total_mv"].isna().sum())
print("预览文件：", preview_path.resolve())

display(merged.head())